# Cleaning the CHROMA dataset for June experiment

In [1]:
import polars as pl
from pathlib import Path
import unicodedata
import re
import os
import shutil
from time import sleep
from datetime import datetime

def clean_folder(path: str, delete_folder: bool = False) -> None:
    if os.path.exists(path):
        shutil.rmtree(path)
        sleep(0.5)
        
    if not delete_folder:
        os.makedirs(path, exist_ok=True)

def clean_column(name):
    replacements = {
        'µ': 'u',
        'μ': 'u',
        '²': '2',
        '³': '3',
    }
    
    for char, replacement in replacements.items():
        name = name.replace(char, replacement)
    
    name = unicodedata.normalize("NFKD", name)
    name = name.encode("ascii", "ignore").decode("ascii")
    name = name.strip()
    name = name.strip('_')
    
    return name

def parse_harmony_dataset(root: Path) -> Path:
    """
    Parses a harmony file directory and yields the cell analysis file

    Args:
        root (Path): Top level folder where all the harmony datasets are located

    Yields:
        Path, str: yields a file path to the cell analysis AND the harmony dataset name
    
    **NOTE:** The harmony file structure is:
        
        raw name/
        ├─ Evaluation/
        │  ├─ PlateResult.txt
        │  ├─ Objects_Population - NoBorderObj.txt (this is analysis file)
        │  ├─ .ExportDone_CellListsToTxt
        ├─ indexfile.txt

    """
    
    for f in root.glob("*/*"):
        if f.is_file() and not f.name.startswith(".") and f.name != "PlateResults.txt":
            # print(f)
            yield f, f.relative_to(root).parts[0]


## Raw dataset formatting

In [3]:
in_path = Path("/home/derfelt/PartchLabFiles/Circadian Rhythms/Live "
               "Study/2026-06-01_20xRep/2026-05-26CP_20xRep_2h_CHROM"
               "A__2026-05-26T16_45_20-Measurement 2")
out_path = Path("/home/derfelt/PartchLabFiles/Circadian Rhythms/Live"
                " Study/formatted_raw_datasets/2026-06-01_CHROMA-For"
                "matted.tsv")

for f, dataset_name in parse_harmony_dataset(in_path):
    print(dataset_name)
    
    with (open(f, "r", encoding='utf-8') as in_file,
          open(out_path, "w", encoding='utf-8') as out_file):
        for i, line in enumerate(in_file):
            curr_line = line.strip()
            
            if i < 9:
                print(f"{curr_line} :: {i}")
            elif i == 9:
                cleaned = "\t".join(clean_column(col) for col in line.strip().split("\t"))
                out_file.write(cleaned + "\n")
            else:
                out_file.write(line.rstrip("\t\n") + "\n")
    print("\n\n")

Evaluation3
Database Name	CSC :: 0
Database Link	http://192.168.0.1/ODA/OdaService.asmx :: 1
Evaluation Signature	4455eda4-822a-4642-81d9-6c7824a0babf :: 2
Plate Name	2026-05-26CP_20xRep_2h :: 3
Measurement	Measurement 2 :: 4
Evaluation	Evaluation3 :: 5
Population	Population - NoBorderObj :: 6
 :: 7
[Data] :: 8





## Formatting tsv files

In [ ]:
drop_cols = [
    "Row",
    "Column",
    "Field",
    "Object No",
    "X",
    "Y",
    "Bounding Box",
    "Position X [um]",
    "Position Y [um]",
    "Compound",
    "Concentration",
    "Cell Type",
    "Cell Count"
]

sampling_interval = 2 # hours
timepoint_col = 'Timepoint'

in_path = Path("/home/derfelt/PartchLabFiles/Circadian Rhythms/Live "
               "Study/formatted_raw_datasets/2026-06-01_CHROMA-Forma"
               "tted.tsv")

out_path = Path("/home/derfelt/PartchLabFiles/Circadian Rhythms/Live "
                "Study/analysis_ready_data")


# schema = pl.scan_csv(in_path, separator="\t").schema.to_python().keys()
# print(*schema, sep="\n")

lf = (
    pl.scan_csv(in_path, 
                separator="\t", 
                null_values=["", "NA", "NaN", "nan", "NULL"])
    .with_columns(
        (
            pl.col("Row").map_elements(lambda x: chr(64+x), return_dtype=pl.String) +
            pl.col("Column").cast(pl.String).str.zfill(2)
        ).alias("WellName")
    )
    .select(["WellName", pl.all().exclude("WellName")])
    .drop(drop_cols)
    .with_columns(
        ((pl.col(timepoint_col) - 1) * sampling_interval).cast(pl.Float64)
    )
)

out_file = out_path / "2026-06-01_20xRep_CHROMA.tsv"
lf.sink_csv(out_file, separator="\t")
